# Split solver: a hand run on the demo fixture

**What this is.** Before `api/rules/allocation.py` is written (C4, Thu 15), the split model is run here on the
synthetic demo snapshot, to check that its rules give sensible answers. It's the "Assignment" in the
[design doc](../../docs/designs/resource-split-and-live-loop.md#the-assignment).

**The model** (design doc, "Split solver"). Each family group (one listing) goes to at most one resource, whole.
A resource needs the capability and access to the area, and has a capacity per help type. Accepted
allocations stay fixed. The objective is lexicographic, solved level by level:

1. **1a:** people covered whose need has a qualified urgent medical requirement;
2. **1b:** other tier-1 people covered (coordinator urgent flag with a reason);
3. **2:** tier-2 person-units covered, `w_evac × evacuation + w_supply × supplies`;
4. **3:** least waiting left uncovered (person-minutes), as the last tie-break only.

Tiers come from the shared `api/rules/tiers.py` (C3), so the coordinator's split panel and the team lists rank
urgency the same way. The solver only *suggests*: it writes nothing, and teams still take work themselves.

**The four checks:** (a) never more than 12 on boat S1; (b) unmet need stays visible, with a reason;
(c) tier 1 is covered before tier 2; (d) the answer changes sensibly when S1 breaks down.
All data is synthetic.

In [1]:
import json, sys
from datetime import datetime
from pathlib import Path

ROOT = Path.cwd().resolve().parents[1] if Path.cwd().name == "notebooks" else Path.cwd().resolve()
sys.path.insert(0, str(ROOT))

from ortools.sat.python import cp_model
from api.rules.tiers import Tier, TierInput, colour, tier

SNAP = json.loads((ROOT / "contract/fixtures/allocationInput.cp3.json").read_text())
print(SNAP["label"])
print(len(SNAP["needs"]), "needs,", len(SNAP["resources"]), "resources,", len(SNAP["accepted"]), "accepted allocation")

Synthetic snapshot for the split solver (C4). Demo, 10:25 Bangkok: T-S has accepted C-01's transport (M-01).
11 needs, 3 resources, 1 accepted allocation


## The snapshot

In [2]:
NOW = datetime.fromisoformat(SNAP["snapshotAt"].replace("Z", "+00:00"))

def needs_of(snap):
    out = []
    for doc in snap["needs"]:
        t = TierInput.from_listing(doc)
        out.append({"id": doc["listingId"], "area": doc["areaId"], "type": doc["assistanceType"], "units": doc["units"],
                    "unit": doc["unit"], "tier": tier(t), "colour": colour(t).value,
                    "waiting_min": int((NOW - t.waiting_since).total_seconds() // 60)})
    return out

print(f"{'need':<12}{'area':<6}{'type':<22}{'units':>6}  tier  colour  waiting")
for n in needs_of(SNAP):
    print(f"{n['id']:<12}{n['area']:<6}{n['type']:<22}{n['units']:>4} {n['unit'][:2]}  {n['tier'].value:<5} {n['colour']:<7} {n['waiting_min']:>4} min")
print()
for r in SNAP["resources"]:
    print(f"{r['resourceId']} ({r['teamId']}, {r['kind']}): {r['capacity']} in {r['areas']} · {r['capacityBasis']}")

need        area  type                   units  tier  colour  waiting
L-C01-MED   P-B   medical                  1 pe  1a    red       10 min
L-C01-EVAC  P-B   evacuation_transport     3 pe  1b    red       10 min
L-C02-EVAC  P-B   evacuation_transport     4 pe  2     yellow    45 min
L-C03-EVAC  P-B   evacuation_transport     2 pe  2     yellow   140 min
L-C04-EVAC  P-B   evacuation_transport     5 pe  2     yellow    75 min
L-C05-EVAC  P-B   evacuation_transport     3 pe  2     yellow   155 min
L-C06-EVAC  P-B   evacuation_transport     4 pe  2     yellow    30 min
L-C08-EVAC  P-B   evacuation_transport     3 pe  2     yellow   115 min
L-C05-SUP   P-B   supplies                 3 pa  2     green    155 min
L-C12-MED   P-A   medical                  1 pe  2     yellow    55 min
L-PA-SUP    P-A   supplies                30 pa  2     green    145 min

S1 (T-S, boat): {'evacuation_transport': 12} in ['P-B', 'P-A'] · 6 seats × 2 trips today
R1 (T-R, truck): {'supplies': 40, 'evacuation_tr

## The model

In [3]:
TYPE_WEIGHT = {"evacuation_transport": "w_evac", "supplies": "w_supply", "medical": "w_evac"}

def solve_split(snap, drop=(), extra=(), weights=None):
    """Suggest which resource covers which family group. Pure: reads the snapshot, writes nothing."""
    weights = {**snap["weights"], **(weights or {})}
    resources = [r for r in snap["resources"] if r["available"] and r["resourceId"] not in drop] + list(extra)
    accepted = [a for a in snap["accepted"] if a["resourceId"] not in drop]  # a broken resource's jobs go back to the list
    needs = needs_of(snap)
    m = cp_model.CpModel()

    x = {}  # (need, resource) -> 1 if this resource covers the whole family group
    for n in needs:
        for r in resources:
            if n["type"] in r["capabilities"] and n["area"] in r["areas"]:
                x[n["id"], r["resourceId"]] = m.NewBoolVar(f"x_{n['id']}_{r['resourceId']}")
    covered = {}
    for n in needs:
        options = [v for (nid, _), v in x.items() if nid == n["id"]]
        covered[n["id"]] = sum(options) if options else 0
        if options:
            m.Add(sum(options) <= 1)  # one resource per family group; never double-booked
    for a in accepted:  # accepted allocations stay as they are
        m.Add(x[a["listingId"], a["resourceId"]] == 1)
    for r in resources:  # capacity, per help type
        for typ, cap in r["capacity"].items():
            m.Add(sum(n["units"] * x[n["id"], r["resourceId"]] for n in needs
                      if (n["id"], r["resourceId"]) in x and n["type"] == typ) <= cap)

    def people(t):
        return sum(n["units"] * covered[n["id"]] for n in needs if n["tier"] == t)

    levels = [
        ("1a urgent medical people covered", people(Tier.T1A), "max"),
        ("1b other urgent people covered", people(Tier.T1B), "max"),
        ("2 weighted units covered (x100)", sum(int(round(100 * weights[TYPE_WEIGHT[n["type"]]])) * n["units"] * covered[n["id"]]
                                              for n in needs if n["tier"] == Tier.T2), "max"),
        ("3 person-minutes left waiting", sum(n["waiting_min"] * n["units"] * (1 - covered[n["id"]]) for n in needs), "min"),
    ]
    solver = cp_model.CpSolver()
    solver.parameters.num_workers = 1  # deterministic: same answer every run
    values = {}
    for name, expr, sense in levels:  # lexicographic: fix each level before the next
        (m.Maximize if sense == "max" else m.Minimize)(expr)
        assert solver.Solve(m) == cp_model.OPTIMAL, name
        values[name] = int(solver.ObjectiveValue())
        m.Add(expr >= values[name]) if sense == "max" else m.Add(expr <= values[name])

    by_id = {r["resourceId"]: r for r in resources}
    assignments, unassigned = [], []
    for n in needs:
        hit = [rid for (nid, rid), v in x.items() if nid == n["id"] and solver.Value(v)]
        if hit:
            assignments.append({**n, "resource": hit[0], "team": by_id[hit[0]]["teamId"],
                                "accepted": any(a["listingId"] == n["id"] for a in accepted)})
        else:
            capable = any(nid == n["id"] for (nid, _) in x)
            unassigned.append({**n, "reason": "capacity_exhausted" if capable else "no_capable_team"})
    return {"inputVersions": snap["inputVersions"], "levels": values, "assignments": assignments, "unassigned": unassigned}


def label(n):
    return {"people": "person", "packs": "pack"}[n["unit"]] if n["units"] == 1 else n["unit"]

def show(s, title):
    print(f"== {title}")
    for name, v in s["levels"].items():
        print(f"   {name}: {v}")
    print("   Assigned:")
    for a in s["assignments"]:
        print(f"     {a['id']:<12}{a['units']:>3} {label(a):<7}→ {a['resource']} ({a['team']})  tier {a['tier'].value}"
              + ("  [accepted, fixed]" if a["accepted"] else ""))
    print("   Not assigned (shown to the coordinator, never hidden):")
    for u in s["unassigned"]:
        print(f"     {u['id']:<12}{u['units']:>3} {label(u):<7}  tier {u['tier'].value}  {u['reason']}")

def used(s, rid, typ="evacuation_transport"):
    return sum(a["units"] for a in s["assignments"] if a["resource"] == rid and a["type"] == typ)

## Run 1: the snapshot as it is (10:25)

In [4]:
base = solve_split(SNAP)
show(base, "Base: S1, R1 and M1 available")

== Base: S1, R1 and M1 available
   1a urgent medical people covered: 1
   1b other urgent people covered: 3
   2 weighted units covered (x100): 3900
   3 person-minutes left waiting: 1360
   Assigned:
     L-C01-MED     1 person → M1 (T-M)  tier 1a
     L-C01-EVAC    3 people → S1 (T-S)  tier 1b  [accepted, fixed]
     L-C02-EVAC    4 people → S1 (T-S)  tier 2
     L-C03-EVAC    2 people → S1 (T-S)  tier 2
     L-C05-EVAC    3 people → S1 (T-S)  tier 2
     L-PA-SUP     30 packs  → R1 (T-R)  tier 2
   Not assigned (shown to the coordinator, never hidden):
     L-C04-EVAC    5 people   tier 2  capacity_exhausted
     L-C06-EVAC    4 people   tier 2  capacity_exhausted
     L-C08-EVAC    3 people   tier 2  capacity_exhausted
     L-C05-SUP     3 packs    tier 2  no_capable_team
     L-C12-MED     1 person   tier 2  capacity_exhausted


**Reading it.** S1 has 12 seats today and 3 are already C-01's (accepted, fixed), so 9 are left for 21 tier-2
people. Several groups add up to 9; the waiting tie-break picks the ones that have waited longest
(C-05 since 07:50, C-03 since 08:05) and fills the last 4 seats with C-02. M1 can make one visit: C-01's urgent
medical need (tier 1a) wins over C-12's routine check. R1 covers P-A's 30 packs but can't enter deep water, so
C-05's packs in P-B have no capable team: that is a real gap, and the split panel says so.

In [5]:
# (a) S1 never carries more than its 12 passenger movements
assert used(base, "S1") <= 12, used(base, "S1")
assert used(base, "S1") == 12

# (b) unmet need stays visible: covered + unassigned = all reviewed demand, each gap with a reason
evac = [n for n in needs_of(SNAP) if n["type"] == "evacuation_transport"]
covered_evac = sum(a["units"] for a in base["assignments"] if a["type"] == "evacuation_transport")
unmet_evac = sum(u["units"] for u in base["unassigned"] if u["type"] == "evacuation_transport")
assert covered_evac + unmet_evac == sum(n["units"] for n in evac) == 24
assert unmet_evac == 12 and all(u["reason"] for u in base["unassigned"])

# (c) tier 1 before tier 2: C-01's medical need gets the only medical visit; C-01's transport is kept
got = {a["id"]: a["resource"] for a in base["assignments"]}
assert got["L-C01-MED"] == "M1" and got["L-C01-EVAC"] == "S1"
assert "L-C12-MED" in {u["id"] for u in base["unassigned"]}
# ... and in general: no tier-1 need is left out while a tier-2 need of the same type holds a resource it could use
t1_out = [u for u in base["unassigned"] if u["tier"] != Tier.T2]
assert not t1_out, t1_out

# the waiting tie-break: of the groups that fill S1's 9 free seats, the longest-waiting ones go
assert {a["id"] for a in base["assignments"] if a["resource"] == "S1"} == {"L-C01-EVAC", "L-C05-EVAC", "L-C03-EVAC", "L-C02-EVAC"}
print("Checks (a), (b), (c) pass")

Checks (a), (b), (c) pass


## Run 2: S1 breaks down (demo step 7, 10:30)

In [6]:
no_s1 = solve_split(SNAP, drop={"S1"})
show(no_s1, "S1 unavailable")

# (d) changes sensibly: nothing can reach P-B by water, so all 24 evacuees are unmet with a reason;
# C-01's medical visit (M-M) is NOT cancelled because the boat failed; P-A's packs are unchanged.
assert all(u["reason"] == "no_capable_team" for u in no_s1["unassigned"] if u["type"] == "evacuation_transport" and u["area"] == "P-B")
assert sum(u["units"] for u in no_s1["unassigned"] if u["type"] == "evacuation_transport") == 24
got = {a["id"]: a["resource"] for a in no_s1["assignments"]}
assert got["L-C01-MED"] == "M1" and got["L-PA-SUP"] == "R1"
print("Check (d) passes")

== S1 unavailable
   1a urgent medical people covered: 1
   1b other urgent people covered: 0
   2 weighted units covered (x100): 3000
   3 person-minutes left waiting: 2315
   Assigned:
     L-C01-MED     1 person → M1 (T-M)  tier 1a
     L-PA-SUP     30 packs  → R1 (T-R)  tier 2
   Not assigned (shown to the coordinator, never hidden):
     L-C01-EVAC    3 people   tier 1b  no_capable_team
     L-C02-EVAC    4 people   tier 2  no_capable_team
     L-C03-EVAC    2 people   tier 2  no_capable_team
     L-C04-EVAC    5 people   tier 2  no_capable_team
     L-C05-EVAC    3 people   tier 2  no_capable_team
     L-C06-EVAC    4 people   tier 2  no_capable_team
     L-C08-EVAC    3 people   tier 2  no_capable_team
     L-C05-SUP     3 packs    tier 2  no_capable_team
     L-C12-MED     1 person   tier 2  capacity_exhausted
Check (d) passes


**Reading it.** With S1 gone, its accepted job (M-01, C-01's transport) goes back to the list rather than staying
on a broken boat. That matches the demo: M-01 is held and relisted by the coordinator, the medical visit is
untouched, and P-A's plan doesn't move.

## Run 3: What if one more boat came? (the split panel's What-if)

In [7]:
S2 = {"resourceId": "S2", "teamId": "T-X", "kind": "boat", "capabilities": ["evacuation_transport"], "areas": ["P-B"],
      "capacity": {"evacuation_transport": 12}, "capacityBasis": "what-if: a second 6-seat boat, 2 trips", "available": True}
more = solve_split(SNAP, extra=[S2])
show(more, "What if: + boat S2 (simulation, not a real team)")
assert sum(a["units"] for a in more["assignments"] if a["type"] == "evacuation_transport") == 24
assert used(more, "S1") <= 12 and used(more, "S2") <= 12
print("A second boat covers all 24 evacuees")

== What if: + boat S2 (simulation, not a real team)
   1a urgent medical people covered: 1
   1b other urgent people covered: 3
   2 weighted units covered (x100): 5100
   3 person-minutes left waiting: 520
   Assigned:
     L-C01-MED     1 person → M1 (T-M)  tier 1a
     L-C01-EVAC    3 people → S1 (T-S)  tier 1b  [accepted, fixed]
     L-C02-EVAC    4 people → S2 (T-X)  tier 2
     L-C03-EVAC    2 people → S1 (T-S)  tier 2
     L-C04-EVAC    5 people → S2 (T-X)  tier 2
     L-C05-EVAC    3 people → S1 (T-S)  tier 2
     L-C06-EVAC    4 people → S1 (T-S)  tier 2
     L-C08-EVAC    3 people → S2 (T-X)  tier 2
     L-PA-SUP     30 packs  → R1 (T-R)  tier 2
   Not assigned (shown to the coordinator, never hidden):
     L-C05-SUP     3 packs    tier 2  no_capable_team
     L-C12-MED     1 person   tier 2  capacity_exhausted
A second boat covers all 24 evacuees


## Weights only move tier 2

In [8]:
# Setting w_evac to 0 must not touch tier 1 (C-01) and must not take seats away from people for no gain.
w0 = solve_split(SNAP, weights={"w_evac": 0.0})
got = {a["id"]: a["resource"] for a in w0["assignments"]}
assert got["L-C01-MED"] == "M1" and got["L-C01-EVAC"] == "S1"
assert w0["levels"]["1a urgent medical people covered"] == base["levels"]["1a urgent medical people covered"]
print("Tier 1 unchanged at w_evac = 0. S1 seats used:", used(w0, "S1"), "of 12")
print("S1 carries:", sorted(a["id"] for a in w0["assignments"] if a["resource"] == "S1"))

Tier 1 unchanged at w_evac = 0. S1 seats used: 11 of 12
S1 carries: ['L-C01-EVAC', 'L-C03-EVAC', 'L-C05-EVAC', 'L-C08-EVAC']


**Finding.** At `w_evac = 0` tier 1 is safe, but S1 carries **11 of 12**: with evacuation coverage worth nothing,
only level 3 (waiting) is left, and it prefers C-08 (waited 115 min, 3 people) over filling the last seat with
C-02. An empty seat while people wait is never what a coordinator means. Proposal for `allocation.py`: keep
each weight at **0.1 or more** in the UI, so seats are always filled first and waiting only breaks ties.

## What this shows, and what's next

- The rules hold on the demo: S1 is never overfilled, the 12 people without a seat stay visible, C-01 comes first,
  and the S1 failure gives the answer the demo script expects.
- Choices made here that the design doc doesn't fix (to confirm):
  - Level 3 counts **person-minutes** (a family of 5 waiting an hour weighs more than one person waiting an hour).
  - Tier-2 **medical** needs use `w_evac`; the doc only names `w_evac` and `w_supply`.
  - When a resource breaks, its accepted jobs go back to the list in the what-if (the coordinator relists them for real).
  - Weights: keep them at 0.1 or more (finding above).
- Next (Thu 15): move `solve_split` into `api/rules/allocation.py` as a pure function `solve_split(snapshot) -> suggestion`,
  with per-need explanation fields for Gemini and a time limit, and add the solver half of T15 to the tests
  (the same `tier()` as the team lists). Open question for the friend: whether a family group may ever be split
  across two resources (here it can't; D6 takes whole groups).